# 01 EDA

目标：读取IEEE-CIS训练数据，形成论文“数据描述性分析”所需的基础结果，包括样本规模、欺诈率、缺失值、时间演化、金额分布和字段族概览。


In [ ]:
from pathlib import Path
import sys
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

PROJECT_ROOT = Path("..").resolve()
sys.path.insert(0, str(PROJECT_ROOT / "src"))

from fraud_detection.utils import load_yaml
from fraud_detection.data import read_ieee_train
from fraud_detection.features import build_feature_system

sns.set_theme(style="whitegrid")
config = load_yaml(PROJECT_ROOT / "configs/base.yaml")
df = read_ieee_train(PROJECT_ROOT / config["data"]["raw_dir"])
df.shape


In [ ]:
target = config["project"]["target"]
profile = pd.DataFrame({
    "Rows": [len(df)],
    "Columns": [df.shape[1]],
    "FraudCount": [int(df[target].sum())],
    "NormalCount": [int((1 - df[target]).sum())],
    "FraudRate": [df[target].mean()],
})
display(profile)
profile.to_csv(PROJECT_ROOT / "outputs/tables/eda_dataset_profile.csv", index=False, encoding="utf-8-sig")


In [ ]:
missing = df.isna().mean().sort_values(ascending=False).rename("MissingRate").reset_index()
missing.columns = ["Feature", "MissingRate"]
display(missing.head(30))
missing.to_csv(PROJECT_ROOT / "outputs/tables/eda_missing_rate.csv", index=False, encoding="utf-8-sig")


In [ ]:
day = df.groupby("relative_day").agg(TransactionCount=(target, "size"), FraudRate=(target, "mean")).reset_index()
hour = df.groupby("hour").agg(TransactionCount=(target, "size"), FraudRate=(target, "mean")).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
sns.lineplot(data=day, x="relative_day", y="FraudRate", ax=axes[0])
axes[0].set_title("Fraud rate by relative day")
sns.barplot(data=hour, x="hour", y="FraudRate", ax=axes[1], color="#4C78A8")
axes[1].set_title("Fraud rate by hour")
plt.tight_layout()
plt.show()


In [ ]:
feature_system = build_feature_system(df, target)
display(feature_system.groupby(["FeatureCategory", "RiskMechanism"]).size().reset_index(name="FeatureCount"))
feature_system.to_csv(PROJECT_ROOT / "outputs/tables/risk_feature_system_eda.csv", index=False, encoding="utf-8-sig")
